# Topic 20 - Data Wrangling

## From many tables to one analysis-ready table

In Topic 19 you gathered data. In Topic 17 you cleaned it. Both of those lessons worked on a single table at a time.

Real data almost never arrives as a single table. A business keeps its orders in one file, its customers in another, and its products in a third, because that is the sensible way to store data. An analyst who wants to answer "which region buys the most cooking oil?" needs all three at once, combined correctly.

**Data wrangling** is the stage that turns several tables in the shape the source keeps them into one table in the shape the analysis needs. It covers three kinds of operation:

| Operation | Question it answers | Main tools |
|---|---|---|
| Combining | How do I put these tables together? | `pd.concat()`, `pd.merge()` |
| Reshaping | How do I turn this table sideways? | `melt()`, `pivot()`, `pivot_table()` |
| Deriving | What new columns does my question need? | `groupby().transform()`, `cut()`, `assign()` |

This lesson works with six real-shaped tables from a small Ghanaian retail distributor. The tables contain four deliberate problems of the kind that make joins go wrong quietly, and a large part of the lesson is learning to detect them.

That word *quietly* is the whole reason this topic exists. A failed join usually does not raise an error. It returns a DataFrame, of a plausible size, full of plausible numbers, that is wrong. Learning to check a join is more important than learning to write one.

## Learning objectives

By the end of this lesson you should be able to:

- explain what makes a table *tidy*, and recognise the common untidy shapes;
- state the **grain** of a table and say why every wrangling decision depends on it;
- stack tables with `pd.concat()`, and know when it is the wrong tool;
- merge two tables with all four join types, and predict the row count before running the code;
- detect and fix the three join failures that do not raise an error: **orphan keys**, **duplicate keys**, and **unclean keys**;
- use `validate=` and `indicator=` to make a silent failure loud;
- reshape a table between wide and long form with `melt()`, `pivot()`, and `pivot_table()`;
- add columns computed within groups using `groupby().transform()` and window functions;
- create categories from numbers with `cut()` and `qcut()`;
- assemble the whole process into one reusable function that takes raw tables and returns an analysis-ready table.

## Wrangling compared with cleaning

Students often ask where cleaning ends and wrangling begins. The distinction is about what you are fixing.

| | Cleaning (Topic 17) | Wrangling (Topic 20) |
|---|---|---|
| The problem | Values are wrong | The table is the wrong shape |
| Works on | One column at a time | Whole tables at a time |
| Typical fix | Strip whitespace, fix a type, drop a duplicate row | Join two tables, melt twelve columns into two |
| Finished when | Every value is trustworthy | One table answers the question |

They are not separate stages in practice, and this lesson will show you why: **you cannot join on a dirty key.** If one table says `Greater Accra` and the other says `greater accra `, the join fails. Cleaning is a prerequisite for wrangling, and you will often discover the cleaning you still owe only when a join produces nothing.

In [1]:
# Run this cell as it is.
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "data_wrangling" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find orders.csv. Expected a data folder containing the six "
        "Topic 20 tables. Looked in: " + ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

print("pandas version:", pd.__version__)
print("data folder:   ", DATA_DIR)
print("files:         ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version: 3.0.3
data folder:    /home/johnny/Projects/ITFY-Ghana/data_wrangling/data
files:          ['customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


---
## 1. Tidy data

Before combining anything, you need a target shape to combine *towards*. That target is called **tidy data**, and it has three rules:

1. **Each variable is a column.**
2. **Each observation is a row.**
3. **Each type of observational unit is its own table.**

Rule 3 is why our data arrives in six files rather than one, and it is correct storage design rather than an inconvenience. Customers and orders are different kinds of thing: a customer exists whether or not they have ordered, and repeating a customer's region on all forty of their order rows would mean forty places to update when they move.

The shape you store data in and the shape you analyse it in are different, and wrangling is the trip between them.

### The three common untidy shapes

| Untidy shape | What it looks like | Fix |
|---|---|---|
| Column headers are values, not variable names | A column per month: `Jan`, `Feb`, `Mar` | `melt()` |
| Several variables crammed in one column | `"Accra-Retail"` in one cell | `str.split()` |
| One observational unit spread across tables you need together | Orders here, customers there | `merge()` |

Our `monthly_targets_wide.csv` has the first problem, on purpose. `Jan` is not a variable; it is a *value* of a variable called `month`.

In [2]:
# A tiny example of the first untidy shape, built by hand so you can see it whole.
untidy = pd.DataFrame({
    "region": ["Greater Accra", "Ashanti", "Northern"],
    "Jan": [9950, 8100, 5200],
    "Feb": [9860, 8300, 5100],
    "Mar": [9990, 8250, 5400],
})
print("UNTIDY - the column headers Jan, Feb, Mar are values of a 'month' variable:")
print(untidy.to_string(index=False))

# melt() moves those headers down into a column of their own.
tidy = untidy.melt(id_vars="region", var_name="month", value_name="target_ghs")
print("\nTIDY - one row per observation, one column per variable:")
print(tidy.to_string(index=False))

print(f"\n{untidy.shape} became {tidy.shape}: 3 rows x 3 months = 9 observations.")
print("The information is identical. Only the shape changed.")

UNTIDY - the column headers Jan, Feb, Mar are values of a 'month' variable:
       region  Jan  Feb  Mar
Greater Accra 9950 9860 9990
      Ashanti 8100 8300 8250
     Northern 5200 5100 5400

TIDY - one row per observation, one column per variable:
       region month  target_ghs
Greater Accra   Jan        9950
      Ashanti   Jan        8100
     Northern   Jan        5200
Greater Accra   Feb        9860
      Ashanti   Feb        8300
     Northern   Feb        5100
Greater Accra   Mar        9990
      Ashanti   Mar        8250
     Northern   Mar        5400

(3, 4) became (9, 3): 3 rows x 3 months = 9 observations.
The information is identical. Only the shape changed.


Look carefully at what `melt()` did, because it is the operation students find least intuitive.

- `id_vars="region"` names the column to **keep as it is**. Everything not listed here gets melted.
- `var_name="month"` names the new column that receives the old **column headers**.
- `value_name="target_ghs"` names the new column that receives the old **cell values**.

Wide data is easier for a person to read. Long data is easier for a computer to group, filter, and join. Reports want wide; analysis wants long.

### Try It 0 - spot the untidy shape

Below is a table of quarterly sales for three products. State which of the three untidy shapes it has, then tidy it with `melt()`. Your result should have the columns `product`, `quarter`, and `sales_ghs`, and 12 rows.

In [ ]:
sales_wide = pd.DataFrame({
    "product": ["Milo Tin 400g", "Frytol 2L", "Key Soap Bar"],
    "Q1": [12400, 9800, 4300],
    "Q2": [13100, 10200, 4100],
    "Q3": [11800, 11500, 4600],
    "Q4": [16900, 12800, 5200],
})
sales_wide

# YOUR CODE HERE
# sales_long = sales_wide.melt(...)
# print(sales_long.shape)
# sales_long

---
## 2. The six tables, and the grain of each

Before joining anything, state what **one row means** in every table you are about to touch. This is called the table's **grain**, and getting it wrong is the single most common cause of a wrong answer in wrangling.

The grain tells you what a join will do. Joining a table whose grain is *one row per order* to one whose grain is *one row per customer* is safe, because many orders point at one customer. Joining two tables that both have many rows per key multiplies them together, which is almost never what anyone wants.

| Table | Grain - one row is... | Key | Rows |
|---|---|---|---|
| `orders.csv` | one order line | `order_id` | ~1,200 |
| `customers.csv` | one customer | `customer_id` | 200 |
| `products.csv` | one product | `product_id` | 60 |
| `regions.csv` | one region | `region` | 16 |
| `returns.csv` | one returned order | `order_id` | ~96 |
| `monthly_targets_wide.csv` | one region, twelve months | `region` | 16 |

`orders` is the **fact table** - the thing that actually happened, the table that grows. The others are **lookup tables**, describing entities that already existed. Almost every wrangling job has this shape: one fact table, several lookups joined onto it.

In [ ]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
targets_wide = pd.read_csv(DATA_DIR / "monthly_targets_wide.csv")

tables = {
    "orders": orders, "customers": customers, "products": products,
    "regions": regions, "returns": returns, "targets_wide": targets_wide,
}
for name, frame in tables.items():
    print(f"{name:14} {frame.shape[0]:>5} rows x {frame.shape[1]:>2} cols   "
          f"{list(frame.columns)}")

In [ ]:
# Confirm the grain rather than trusting the table above. A key that is unique
# is a key you can safely join ON; a key that repeats is one you join FROM.
def check_grain(frame, key, name):
    total = len(frame)
    unique = frame[key].nunique()
    duplicated = total - unique
    verdict = "unique - safe to join onto" if duplicated == 0 else \
              f"NOT UNIQUE - {duplicated} duplicate key value(s)"
    print(f"{name:14} {key:12} {unique:>5} unique / {total:>5} rows   {verdict}")


check_grain(orders, "order_id", "orders")
check_grain(customers, "customer_id", "customers")
check_grain(products, "product_id", "products")
check_grain(regions, "region", "regions")
check_grain(returns, "order_id", "returns")

Read that output before going on. Five of those checks pass and **one fails**: `products` has 61 rows but only 60 unique `product_id` values.

That is the first of the four deliberate problems in this data. We will come back to it in section 5, where you will see exactly how much damage one duplicated key does. For now, notice that finding it took one cell, ran before any joining, and needed no special tools.

### Try It 1 - inspect a table you have not looked at yet

Print the first five rows of `returns`, then answer with code:

1. How many orders were returned?
2. What proportion of all orders is that?
3. What are the distinct return reasons, and how often does each occur?

In [ ]:
# YOUR CODE HERE
# returns.head()
# print(len(returns), "of", len(orders), "orders were returned")

---
## 3. Stacking tables with `pd.concat()`

`concat` glues tables together without matching anything up. Use it when your tables are **the same kind of thing split into pieces** - January and February in separate files, or one CSV per branch.

A very common real situation: a system exports one file per month and you need the year.

In [ ]:
# Split the orders into two halves, as though they had arrived as two files.
orders["order_date"] = pd.to_datetime(orders["order_date"])
first_half = orders[orders["order_date"] < "2023-07-01"]
second_half = orders[orders["order_date"] >= "2023-07-01"]
print(f"first_half:  {first_half.shape}")
print(f"second_half: {second_half.shape}")

# Stack them back into one table.
recombined = pd.concat([first_half, second_half], ignore_index=True)
print(f"recombined:  {recombined.shape}")
print(f"Row count preserved: {len(recombined) == len(orders)}")

### Why `ignore_index=True` matters

Each half kept its original row labels, so stacking them produces a table whose index runs `0, 1, 2, ... 600, 0, 1, 2, ...` - with duplicates. `ignore_index=True` throws the old labels away and numbers the result from zero.

Leave it out and most things still work, until you use `.loc[0]` and get two rows back instead of one.

In [ ]:
careless = pd.concat([first_half, second_half])          # no ignore_index
print("Index has duplicates:", careless.index.duplicated().any())
print("How many rows does .loc[0] return?", len(careless.loc[[0]]))

careful = pd.concat([first_half, second_half], ignore_index=True)
print("\nWith ignore_index=True:")
print("Index has duplicates:", careful.index.duplicated().any())
print("How many rows does .loc[0] return?", len(careful.loc[[0]]))

In [ ]:
# concat matches on COLUMN NAMES, not position. A mismatched name becomes a new
# column full of NaN rather than an error - another silent failure to watch for.
branch_a = pd.DataFrame({"order_id": ["O1", "O2"], "amount_ghs": [120, 80]})
branch_b = pd.DataFrame({"order_id": ["O3"], "amount": [45]})   # different name

stacked = pd.concat([branch_a, branch_b], ignore_index=True)
print(stacked.to_string(index=False))
print("\nNo error was raised, but the result has 3 columns instead of 2,")
print("and every value is in the wrong place half the time.")
print("\nThe fix is to rename BEFORE stacking:")
fixed = pd.concat(
    [branch_a, branch_b.rename(columns={"amount": "amount_ghs"})],
    ignore_index=True,
)
print(fixed.to_string(index=False))

### `keys=` records where each row came from

If you need to remember which file a row arrived in, `keys=` labels each piece.

In [ ]:
labelled = pd.concat(
    [first_half, second_half],
    keys=["H1", "H2"],
    names=["half", "row_number"],
).reset_index(level="half")

print(labelled[["half", "order_id", "order_date", "quantity"]].head(3).to_string(index=False))
print("...")
print(labelled[["half", "order_id", "order_date", "quantity"]].tail(3).to_string(index=False))
print("\nRows per half:")
print(labelled["half"].value_counts().to_string())

### Try It 2 - stack with a mismatch

Two branches sent their order counts, but they do not agree on column names.

1. Stack them with `pd.concat()` as they are, and print the result. How many columns did you get, and how many `NaN` values?
2. Fix it by renaming, stack again, and confirm you now have exactly three columns and no missing values.

In [ ]:
tema_branch = pd.DataFrame({
    "branch": ["Tema", "Tema"],
    "month": ["Jan", "Feb"],
    "orders_count": [82, 91],
})
ho_branch = pd.DataFrame({
    "branch": ["Ho", "Ho"],
    "month": ["Jan", "Feb"],
    "num_orders": [34, 39],          # a different name for the same variable
})

# YOUR CODE HERE

---
## 4. Merging: the four join types

`concat` stacks. `merge` **matches**. It looks up rows in one table using a key found in another, and it is the single most important operation in this lesson.

Every merge needs three decisions:

1. **Which key?** The column whose values appear in both tables.
2. **Which rows to keep?** This is the `how=` argument, and it is the join type.
3. **What should the row count be afterwards?** Decide this *before* you run it, then check.

That third decision is the one people skip. If you cannot say in advance roughly how many rows you expect, you cannot tell whether the merge worked.

### The four join types

| `how=` | Keeps | Use when |
|---|---|---|
| `"inner"` | Only keys found in **both** tables | You want complete records only, and are willing to lose the rest |
| `"left"` | **All** rows of the left table | The left table is your fact table and you must not lose any of it |
| `"right"` | All rows of the right table | Rare - usually clearer to swap the tables and use `left` |
| `"outer"` | Everything from **both** | You are reconciling two lists and want to see what each one is missing |

**`left` is the default choice for an analyst.** Your fact table is the thing that happened, and a lookup failing to match is a data problem you need to see, not rows you should silently discard.

In [ ]:
# A tiny worked example, small enough to check by eye.
left_table = pd.DataFrame({
    "order_id": ["O1", "O2", "O3"],
    "customer_id": ["C1", "C2", "C9"],      # C9 is not in the right table
})
right_table = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],      # C3 has no orders
    "region": ["Greater Accra", "Ashanti", "Volta"],
})
print("LEFT (orders):");  print(left_table.to_string(index=False))
print("\nRIGHT (customers):"); print(right_table.to_string(index=False))

for how in ["inner", "left", "right", "outer"]:
    result = left_table.merge(right_table, on="customer_id", how=how)
    print(f"\nhow='{how}'  ->  {len(result)} rows")
    print(result.to_string(index=False))

Work through that output slowly, because the four results answer four different questions.

- **`inner`** gives 2 rows. Order `O3` disappeared, because customer `C9` is unknown. Nothing warned you. **The rows you lose to an inner join leave no trace.**
- **`left`** gives 3 rows. `O3` survives with `region` as `NaN`. The problem is now visible, which is exactly why this is the safe default.
- **`right`** gives 3 rows: the two matches plus customer `C3`, who has never ordered, with `order_id` as `NaN`.
- **`outer`** gives 4 rows - every order and every customer, matched where possible. This is the reconciliation view: it shows both the unknown customer and the customer with no orders at once.

Notice that `inner` and `left` returned different row counts from the same two tables. Choosing a join type is not a style preference; it changes your answer.

---
## 5. Three ways a merge fails quietly

Now the real data. Our six tables contain four deliberate problems, and three of them are join failures that raise no error at all.

### Hazard 1 - orphan keys

An **orphan key** is a key value in the fact table with no match in the lookup. Here, some orders were placed by customer ids that do not appear in `customers.csv` - a completely ordinary situation caused by a customer record being deleted, or two systems being exported at different times.

In [ ]:
# Count the orphans BEFORE merging. A set difference is the cheapest possible check.
order_customer_ids = set(orders["customer_id"])
known_customer_ids = set(customers["customer_id"])
orphans = order_customer_ids - known_customer_ids

affected_rows = orders["customer_id"].isin(orphans).sum()
print(f"Unknown customer_id values: {len(orphans)}  ->  {sorted(orphans)}")
print(f"Order rows affected:        {affected_rows} of {len(orders)}")

In [ ]:
# Now watch what each join type does with them.
orders["revenue_ghs"] = orders["quantity"] * orders["unit_price_ghs"]
true_revenue = orders["revenue_ghs"].sum()

inner_join = orders.merge(customers, on="customer_id", how="inner")
left_join = orders.merge(customers, on="customer_id", how="left")

print(f"orders               {len(orders):>5} rows   GHS {true_revenue:>12,.2f}")
print(f"merge(how='inner')   {len(inner_join):>5} rows   "
      f"GHS {inner_join['revenue_ghs'].sum():>12,.2f}")
print(f"merge(how='left')    {len(left_join):>5} rows   "
      f"GHS {left_join['revenue_ghs'].sum():>12,.2f}")

lost = true_revenue - inner_join["revenue_ghs"].sum()
print(f"\nThe inner join silently discarded {len(orders) - len(inner_join)} orders")
print(f"worth GHS {lost:,.2f} - about {lost / true_revenue:.1%} of total revenue.")
print("No error. No warning. Just a smaller, wrong answer.")

### `indicator=True` makes the invisible visible

Passing `indicator=True` adds a `_merge` column recording, for every row, whether the key was found in the left table only, the right table only, or both. It costs nothing and turns a silent failure into a printable number.

In [ ]:
checked = orders.merge(customers, on="customer_id", how="outer", indicator=True)
print("Where did each row come from?")
print(checked["_merge"].value_counts().to_string())
print()
print("  both       - an order whose customer is known. This is the healthy case.")
print("  left_only  - an order placed by an unknown customer (our orphans).")
print("  right_only - a customer who has never ordered.")

print("\nThe orphaned orders:")
print(checked[checked["_merge"] == "left_only"]
      [["order_id", "customer_id", "order_date", "revenue_ghs"]]
      .head(5).to_string(index=False))

Having found them, you must **decide** what to do, and record the decision:

- **Keep them** with a `left` join and `region` as `NaN` - right when the totals must be complete. Revenue is revenue even if you cannot attribute it to a region.
- **Drop them** with an `inner` join - right when every row must be fully attributed, and you have said so in writing.
- **Investigate** - right when 27 orphans out of 1,200 might mean a broken export rather than a few deleted customers.

What is never acceptable is arriving at one of these by accident because you typed `how="inner"` without thinking.

### Hazard 2 - duplicate keys, and the row explosion

This is the most dangerous failure in the lesson, because the result looks completely normal.

A merge matches **every** left row against **every** matching right row. If a key appears twice on the right, each left row matching it comes back twice. Our `products` table has one `product_id` listed twice, under two different suppliers.

In [ ]:
duplicated_ids = products.loc[products["product_id"].duplicated(keep=False)]
print("The duplicated product:")
print(duplicated_ids.to_string(index=False))
print("\nThe product_id is the same. The supplier differs, so the two rows are")
print("not exact duplicates and drop_duplicates() alone will not remove them.")

In [ ]:
naive = orders.merge(products, on="product_id", how="left")

print(f"orders before merge: {len(orders):>5} rows")
print(f"orders after merge:  {len(naive):>5} rows")
print(f"rows created:        {len(naive) - len(orders):>5}")

naive["revenue_ghs"] = naive["quantity"] * naive["unit_price_ghs"]
inflated = naive["revenue_ghs"].sum()
print(f"\ntrue revenue:      GHS {true_revenue:>12,.2f}")
print(f"revenue after:     GHS {inflated:>12,.2f}")
print(f"overstated by:     GHS {inflated - true_revenue:>12,.2f}  "
      f"({inflated / true_revenue - 1:.2%})")

Read those numbers again. The merge invented **21 orders that never happened** and overstated revenue by **GHS 11,309.91**.

Now consider how this looks in a report. The row count went from 1,200 to 1,221 - a 1.75% rise that nobody notices. Revenue rose about 5%, which does not look like a bug; it looks like a good year. Every individual row is a real product with a real price. There is no `NaN`, no error, and no warning.

This is the failure that ends up in board papers. The only reliable defence is to state your expectation and have pandas enforce it.

### `validate=` turns the assumption into a check

`validate=` tells pandas what relationship you believe the two tables have. If you are wrong, it raises an exception instead of returning bad data.

| `validate=` | Means | Typical use |
|---|---|---|
| `"m:1"` | Many left rows to one right row | **Joining a lookup onto a fact table. Use this constantly.** |
| `"1:1"` | One to one | Merging two tables that should be the same list |
| `"1:m"` | One left row to many right rows | Expanding a parent into its children |
| `"m:m"` | Many to many | Almost always a mistake - the row explosion is the point |

In [ ]:
# We believe many orders point at one product. Ask pandas to enforce it.
try:
    safe = orders.merge(products, on="product_id", how="left", validate="m:1")
    print("Merge succeeded - the relationship was as expected.")
except pd.errors.MergeError as error:
    print("MergeError raised, exactly as we want:")
    print(" ", error)
    print("\nThe merge refused to run rather than returning inflated numbers.")

In [ ]:
# Fix the lookup, then merge again with the same guard in place.
products_clean = products.drop_duplicates(subset="product_id", keep="first")
print(f"products:       {len(products)} rows")
print(f"products_clean: {len(products_clean)} rows "
      f"({len(products) - len(products_clean)} duplicate key removed)")

orders_products = orders.merge(
    products_clean, on="product_id", how="left", validate="m:1"
)
print(f"\nmerge with validate='m:1' succeeded: {len(orders_products)} rows")
print(f"Row count preserved: {len(orders_products) == len(orders)}")
print(f"Revenue correct:     "
      f"{orders_products['revenue_ghs'].sum().round(2) == round(true_revenue, 2)}")

Note what `keep="first"` did: it chose the **Tamale Distribution Hub** row and discarded the **Kumasi Wholesale Ltd** row, purely because it came first in the file.

That is a real decision about the data, not a technicality. If the two rows had different `unit_cost_ghs` values, `keep="first"` would silently pick one cost and change every profit figure that follows. In a real project the right move is to ask whoever owns the product list which row is correct. When you cannot ask, record the assumption - it belongs in the manifest you learned to write in Topic 19.

### Hazard 3 - an unclean key

The third failure is the one that connects this lesson back to Topic 17. Both tables have the column, both tables have the values, and the join still produces nothing - because the values are not *identical*.

In [ ]:
naive_regions = customers.merge(regions, on="region", how="left")
unmatched = naive_regions["zone"].isna().sum()

print(f"customers: {len(customers)} rows")
print(f"customers with no zone after the merge: {unmatched} "
      f"({unmatched / len(customers):.0%})")

# repr() rather than print() for the values, so the spaces become visible.
# A bare Central and a ' Central' look identical on screen otherwise.
failed = sorted(set(customers["region"]) - set(regions["region"]))
print(f"\n{len(failed)} distinct spellings failed to match, "
      f"for only 16 real regions:\n")
for row_start in range(0, len(failed), 3):
    print("   " + "".join(f"{value!r:<22}" for value in failed[row_start:row_start + 3]))

Look at those values. Every one is a real Ghanaian region, spelled recognisably, and every one fails to match because a string comparison is **exact**. To Python, `"Ashanti"` and `"ashanti "` have nothing in common at all.

There are only three problems in that list, repeated in different combinations:

| Problem | Example | Fix |
|---|---|---|
| Stray leading or trailing space | `' Central'`, `'Western '` | `.str.strip()` |
| Doubled internal space | `'Upper  East'` | `.str.replace(r"\s+", " ", regex=True)` |
| Inconsistent capitalisation | `'BONO'`, `'volta'` | `.str.title()` |

Sixteen regions produced 31 broken spellings. That ratio is normal for data typed by people.

The merge did not fail. It returned 200 rows, as requested, with a column that is 20% empty. If you then grouped by `zone`, those customers would simply vanish from the report.

The fix is Topic 17 work, done before the join rather than after it.

In [ ]:
def clean_region(series):
    """Normalise a region column so it can be used as a join key."""
    return (series.str.strip()                       # remove leading/trailing spaces
                  .str.replace(r"\s+", " ", regex=True)   # collapse double spaces
                  .str.title())                      # consistent capitalisation


customers_clean = customers.copy()
customers_clean["region"] = clean_region(customers_clean["region"])

fixed = customers_clean.merge(regions, on="region", how="left", validate="m:1")
print(f"unmatched before cleaning: {unmatched}")
print(f"unmatched after cleaning:  {fixed['zone'].isna().sum()}")

print("\nCustomers per zone, now that every row matches:")
print(fixed["zone"].value_counts().to_string())

### The merge checklist

Three hazards, three checks. Run them every time, in this order:

1. **Before merging**, compare the key sets: `set(left[key]) - set(right[key])`. This finds orphans.
2. **Before merging**, check the right key is unique: `right[key].duplicated().any()`. This finds explosions.
3. **When merging**, pass `validate="m:1"` and, when you are unsure, `indicator=True`.
4. **After merging**, compare the row count with the row count before. It should be identical for an `m:1` left join.

Four lines of code. They would have caught every problem in this section.

### Try It 3 - merge the returns table safely

`returns` covers only about 8% of orders, so this is a natural `left` join.

1. Before merging, check that `order_id` is unique in `returns`.
2. Merge `returns` onto `orders` with a left join, `validate="m:1"`, and `indicator=True`.
3. Confirm the row count did not change.
4. Print the `_merge` value counts. How many orders were returned, and how many were not?
5. Create a boolean column `was_returned` that is `True` when the order appears in `returns`.
6. What percentage of revenue was returned?

In [ ]:
# YOUR CODE HERE
# print("order_id unique in returns:", not returns["order_id"].duplicated().any())
# with_returns = orders.merge(returns, on="order_id", how="left",
#                             validate="m:1", indicator=True)

---
## 6. Reshaping: wide and long

A table can hold identical information in two shapes.

**Long** (also called tidy or stacked) has one row per observation and a column naming the variable:

```text
region          month   target_ghs
Greater Accra   Jan          9950
Greater Accra   Feb          9860
```

**Wide** (also called pivoted) spreads one variable across the columns:

```text
region            Jan     Feb
Greater Accra    9950    9860
```

Neither is better in the abstract. They serve different purposes:

| | Long | Wide |
|---|---|---|
| Good for | Grouping, filtering, joining, plotting | Reading, comparing side by side, reports |
| Adding a month means | Adding rows | Adding a **column**, and changing your code |
| Missing data shows as | An absent row | An explicit `NaN` cell |
| Machines prefer | This one | |
| People prefer | | This one |

**Analyse in long form. Present in wide form.** That single sentence covers most real decisions about which to use.

In [ ]:
# Our targets arrived wide - one column per month.
print("monthly_targets_wide.csv as stored:")
print(targets_wide.head(3).to_string(index=False))
print(f"\nShape: {targets_wide.shape} - 16 regions, 12 month columns plus the region name")

In [ ]:
# melt() turns it long: the 12 month HEADERS become values in a 'month' column.
targets = targets_wide.melt(
    id_vars="region",          # keep as it is
    var_name="month",          # new column for the old headers
    value_name="target_ghs",   # new column for the old cell values
)
print(f"{targets_wide.shape} melted into {targets.shape}")
print(f"16 regions x 12 months = {16 * 12} rows\n")
print(targets.head(4).to_string(index=False))
print("...")
print(targets.tail(4).to_string(index=False))

The long version is what makes the targets *usable*. In wide form, "what is the Ashanti target for March?" needs you to know that `Mar` is a column name. In long form it is an ordinary filter, identical to every other filter you have ever written, and the table can now be merged onto anything else that has a region and a month.

### `pivot_table()` goes the other way

`pivot_table()` turns long back into wide, **aggregating** as it goes. This is the operation that produces the summary table a manager actually wants to look at.

In [ ]:
# Build an analysis table with the region attached, cleaned and validated.
# Note the order: the zone is attached to CUSTOMERS first, then customers are
# attached to orders. Joining regions straight onto orders would not work,
# because orders has no region column of its own.
customers_with_zone = customers_clean.merge(
    regions, on="region", how="left", validate="m:1"
)
orders_full = (
    orders
    .merge(customers_with_zone, on="customer_id", how="left", validate="m:1")
    .merge(products_clean, on="product_id", how="left", validate="m:1")
)
orders_full["month"] = orders_full["order_date"].dt.strftime("%b")
print(f"orders_full: {orders_full.shape} - row count preserved: "
      f"{len(orders_full) == len(orders)}")

# Long -> wide, summing revenue for each region and month.
revenue_matrix = orders_full.pivot_table(
    index="region",
    columns="month",
    values="revenue_ghs",
    aggfunc="sum",
    fill_value=0,
)

# Calendar order, because pivot_table sorts the columns alphabetically:
# Apr, Aug, Dec, Feb ... which is never what anyone wants.
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
revenue_matrix = revenue_matrix[month_order]

print("\nRevenue by region and month (GHS):")
print(revenue_matrix.round(0).head(6).to_string())

That alphabetical-column trap is worth remembering. `pivot_table()` sorts columns as strings, so months come out `Apr, Aug, Dec, Feb...`. The table is not wrong, but it is unreadable, and a reader who glances at it will misread the trend. Always reorder the columns after pivoting on anything with a natural order.

### `pivot()` versus `pivot_table()`

| | `pivot()` | `pivot_table()` |
|---|---|---|
| Aggregates | No | Yes, via `aggfunc` |
| Duplicate index/column pairs | Raises an error | Combines them |
| Use when | Your data is already one row per cell | Anything else |

`pivot()` is stricter, and that strictness is occasionally useful: if you believe you have exactly one value per combination, `pivot()` will tell you when you are wrong. For summarising raw records, you want `pivot_table()`.

In [ ]:
# pivot() on data with duplicates raises, rather than quietly picking one.
try:
    orders_full.pivot(index="region", columns="month", values="revenue_ghs")
except ValueError as error:
    print("pivot() refused, and it is right to:")
    print(" ", str(error)[:110], "...")
    print("\nThere are many orders per region per month, so there is no single")
    print("value to put in each cell. pivot_table(aggfunc='sum') answers the")
    print("question that pivot() cannot.")

### Joining the two shapes together

Now that `targets` is long, it can be merged onto the monthly actuals - which is the whole reason we melted it. This is the payoff of the section: **reshaping is what makes a join possible.**

In [ ]:
# Actual revenue per region per month, in long form.
actuals = (orders_full
           .groupby(["region", "month"], as_index=False)["revenue_ghs"]
           .sum()
           .rename(columns={"revenue_ghs": "actual_ghs"}))

# Merge on TWO keys at once, by passing a list.
performance = actuals.merge(targets, on=["region", "month"], how="left", validate="1:1")
performance["achieved_pct"] = (performance["actual_ghs"] / performance["target_ghs"] * 100).round(1)

print(f"actuals: {actuals.shape}   targets: {targets.shape}   merged: {performance.shape}")
print("\nWorst-performing region-months against target:")
print(performance.nsmallest(5, "achieved_pct").to_string(index=False))

Note `on=["region", "month"]`. A key does not have to be a single column: here a row is only identified by a region *and* a month together, so both are needed. Merging on `region` alone would match each region's actuals to all twelve of its targets and produce twelve times too many rows.

Note also `validate="1:1"` - each region-month appears once on each side, so anything else is a bug worth stopping for.

### `stack()` and `unstack()`

These do the same wide/long conversion but work on the **index** rather than on columns. You will meet them mostly as a quick way to flatten the result of a two-column `groupby`.

In [ ]:
by_zone_channel = orders_full.groupby(["zone", "channel"])["revenue_ghs"].sum()
print("A grouped Series with two index levels (long):")
print(by_zone_channel.head(5).to_string())

print("\n.unstack() pushes the inner level up into columns (wide):")
print(by_zone_channel.unstack().round(0).to_string())

print("\n.stack() puts it back (long again):")
print(by_zone_channel.unstack().stack().head(3).to_string())

### Try It 4 - reshape in both directions

1. Build a wide table of **order counts** by `category` (rows) and `channel` (columns) using `pivot_table()` with `aggfunc="size"`.
2. Which category-channel combination has the most orders?
3. Now melt your wide table back into long form with columns `category`, `channel`, and `order_count`.
4. Confirm the total order count is the same in both shapes.

In [ ]:
# YOUR CODE HERE
# counts_wide = orders_full.pivot_table(index=..., columns=..., aggfunc="size", fill_value=0)

---
## 7. Deriving columns within groups

Some questions cannot be answered by a single row on its own. "Is this order large *for its region*?" needs the region's average, which lives in other rows.

This is where `transform()` comes in, and where beginners most often reach for a loop.

### `agg` versus `transform` versus `apply`

| Method | Returns | Use for |
|---|---|---|
| `.agg()` | **One row per group** | A summary table |
| `.transform()` | **One row per original row** | A new column on the original table |
| `.apply()` | Whatever your function returns | Anything the other two cannot do - and it is slower |

The distinction is the shape of the result. `agg` collapses; `transform` broadcasts the group's answer back out to every row of that group.

In [ ]:
regional = orders_full[["order_id", "region", "revenue_ghs"]].copy()

# agg: 16 rows, one per region.
summary = regional.groupby("region")["revenue_ghs"].sum()
print(f".agg / .sum()   -> {len(summary)} rows (one per region)")

# transform: 1,200 rows, the region's total repeated on each of its rows.
regional["region_total"] = regional.groupby("region")["revenue_ghs"].transform("sum")
print(f".transform()    -> {len(regional)} rows (one per order)\n")

# Which is exactly what you need to compute a share.
regional["pct_of_region"] = (regional["revenue_ghs"] / regional["region_total"] * 100).round(2)
# Note the nullable "Int64" rather than plain int. The orphan orders from
# section 5 have no region, so pandas cannot rank them within one - their rank
# is genuinely missing, and a plain int column cannot hold a missing value.
regional["rank_in_region"] = (regional.groupby("region")["revenue_ghs"]
                                      .rank(ascending=False, method="min")
                                      .astype("Int64"))

print(regional.sort_values(["region", "rank_in_region"]).head(6).to_string(index=False))
print(f"\nOrders with no rank (the orphans again): "
      f"{regional['rank_in_region'].isna().sum()}")

### Window functions: looking at neighbouring rows

A **window function** computes a value from a row's neighbours: the running total so far, the previous month, a moving average. In pandas these are ordinary methods, and they all depend on the table being **sorted correctly first**.

| Method | Gives |
|---|---|
| `.cumsum()` | Running total |
| `.shift(1)` | The previous row's value |
| `.pct_change()` | Change from the previous row, as a proportion |
| `.rolling(n).mean()` | Moving average over `n` rows |
| `.rank()` | Position within the group |

Sorting is not optional. `shift(1)` returns whatever row happens to sit above, so on unsorted data "last month" might be August.

In [ ]:
monthly = (orders_full
           .groupby(orders_full["order_date"].dt.to_period("M"))["revenue_ghs"]
           .sum()
           .reset_index())
monthly["order_date"] = monthly["order_date"].astype(str)
monthly = monthly.sort_values("order_date").reset_index(drop=True)   # sort FIRST

monthly["running_total"] = monthly["revenue_ghs"].cumsum().round(2)
monthly["previous_month"] = monthly["revenue_ghs"].shift(1).round(2)
monthly["change_pct"] = (monthly["revenue_ghs"].pct_change() * 100).round(1)
monthly["rolling_3m"] = monthly["revenue_ghs"].rolling(3).mean().round(2)

print(monthly.to_string(index=False))
print("\nThe first row's previous_month and change_pct are NaN, and rolling_3m")
print("is NaN until row 3. That is correct: there is no earlier data to use.")
print("Filling those with 0 would invent a month in which nothing was sold.")

---
## 8. Turning numbers into categories

The last wrangling move is **binning**: converting a continuous number into a small set of labels, so it can be grouped and counted.

| Tool | Cuts by | Use when |
|---|---|---|
| `pd.cut()` | Boundaries **you** choose | The thresholds have a business meaning |
| `pd.qcut()` | Equal-sized groups (quantiles) | You want "the top 25%" regardless of value |
| `np.where()` | A single condition | A two-way split |
| `.map()` | A dictionary lookup | Recoding existing labels |

`cut` and `qcut` answer different questions, and mixing them up produces confusing reports. `cut` says "orders over GHS 500 are large" - a fixed, meaningful line. `qcut` says "the largest quarter of orders" - a relative line that moves as the data changes.

In [ ]:
import numpy as np

banded = orders_full[["order_id", "region", "revenue_ghs"]].copy()

# cut: boundaries chosen for business reasons. Note right=False so the bands
# are [0, 50), [50, 200) ... and a value of exactly 50 lands in "Small".
banded["size_band"] = pd.cut(
    banded["revenue_ghs"],
    bins=[0, 50, 200, 500, np.inf],
    labels=["Micro", "Small", "Medium", "Large"],
    right=False,
)

# qcut: four groups of equal SIZE, whatever the values happen to be.
banded["quartile"] = pd.qcut(banded["revenue_ghs"], q=4,
                             labels=["Q1", "Q2", "Q3", "Q4"])

# np.where: a simple two-way flag.
banded["is_large"] = np.where(banded["revenue_ghs"] >= 500, "Yes", "No")

print("cut() - fixed boundaries, so group sizes vary:")
print(banded["size_band"].value_counts().sort_index().to_string())
print("\nqcut() - equal group sizes, so boundaries vary:")
print(banded["quartile"].value_counts().sort_index().to_string())
print("\nThe quartile boundaries qcut chose (GHS):")
print(pd.qcut(banded["revenue_ghs"], q=4).cat.categories)

### Try It 5 - band the customers

1. Using `orders_full`, compute total revenue per customer.
2. Use `pd.qcut()` to split customers into three tiers: `Bronze`, `Silver`, `Gold`.
3. How many customers are in each tier, and what revenue does each tier account for?
4. Explain in one sentence why the three tiers contain roughly equal *numbers* of customers but very different *amounts* of revenue.

In [ ]:
# YOUR CODE HERE

---
## 9. Putting it together: one reusable function

You have now used every tool in the lesson. The final step is to stop running cells one at a time and write the whole pipeline as a **function**.

This matters for three reasons. It can be re-run when next month's data arrives. It can be read by somebody checking your work. And it fails loudly when an assumption breaks, instead of producing a quietly wrong table.

Notice how the function is built: every merge carries `validate=`, every cleaning step happens before the join that needs it, and the function checks its own row count at the end.

In [ ]:
def build_analysis_table(orders, customers, products, regions, returns):
    """Combine the five raw tables into one analysis-ready table.

    Returns one row per order, with customer, region, product, and return
    details attached, plus derived revenue and profit columns.
    """
    expected_rows = len(orders)

    # --- 1. Clean the keys BEFORE joining on them -------------------------
    customers = customers.copy()
    customers["region"] = (customers["region"].str.strip()
                                              .str.replace(r"\s+", " ", regex=True)
                                              .str.title())

    # --- 2. Remove the duplicate product key ------------------------------
    products = products.drop_duplicates(subset="product_id", keep="first")

    # --- 3. Attach the region's zone to each customer ---------------------
    customers = customers.merge(regions, on="region", how="left", validate="m:1")

    # --- 4. Join the lookups onto the fact table --------------------------
    #     how="left" keeps every order, including the orphans.
    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))

    # --- 5. Flag returns without losing any orders ------------------------
    returned_ids = set(returns["order_id"])
    table["was_returned"] = table["order_id"].isin(returned_ids)

    # --- 6. Derive the columns the analysis needs -------------------------
    table["order_date"] = pd.to_datetime(table["order_date"])
    table["month"] = table["order_date"].dt.strftime("%b")
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    table["size_band"] = pd.cut(table["revenue_ghs"],
                                bins=[0, 50, 200, 500, float("inf")],
                                labels=["Micro", "Small", "Medium", "Large"],
                                right=False)

    # --- 7. Check the work before handing it back -------------------------
    if len(table) != expected_rows:
        raise ValueError(
            f"Row count changed during wrangling: {expected_rows} -> {len(table)}. "
            "A merge has duplicated rows."
        )
    return table.sort_values("order_date").reset_index(drop=True)

In [ ]:
analysis = build_analysis_table(
    pd.read_csv(DATA_DIR / "orders.csv"),
    pd.read_csv(DATA_DIR / "customers.csv"),
    pd.read_csv(DATA_DIR / "products.csv"),
    pd.read_csv(DATA_DIR / "regions.csv"),
    pd.read_csv(DATA_DIR / "returns.csv"),
)
print(f"Analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")
print(f"Row count matches orders: {len(analysis) == len(orders)}")
print(f"Total revenue: GHS {analysis['revenue_ghs'].sum():,.2f}")
print(f"Total profit:  GHS {analysis['profit_ghs'].sum():,.2f}")
print(f"Orders with no known customer: {analysis['region'].isna().sum()}")
print(f"Returned orders: {analysis['was_returned'].sum()}")
print("\nColumns now available:")
print(list(analysis.columns))

In [ ]:
# The payoff: questions that needed five tables now need one line each.
print("Profit by zone:")
print(analysis.groupby("zone")["profit_ghs"].sum().round(2).sort_values(ascending=False).to_string())

print("\nTop 5 categories by revenue:")
print(analysis.groupby("category")["revenue_ghs"].sum().round(2)
              .sort_values(ascending=False).head(5).to_string())

print("\nReturn rate by channel:")
print((analysis.groupby("channel")["was_returned"].mean() * 100).round(1).to_string())

In [ ]:
# Two charts, using the shapes we built. Line for change over time, bar for
# comparing categories - the same rule as the rest of the course.
monthly_revenue = (analysis.groupby(analysis["order_date"].dt.to_period("M"))
                           ["revenue_ghs"].sum())
monthly_revenue.index = monthly_revenue.index.astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(monthly_revenue.index, monthly_revenue.values,
             marker="o", linewidth=2, color="#c0392b")
axes[0].set_title("Revenue rose sharply in November and December 2023")
axes[0].set_xlabel("Month of 2023")
axes[0].set_ylabel("Revenue (GHS)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(True, alpha=0.3)

zone_profit = analysis.groupby("zone")["profit_ghs"].sum().sort_values(ascending=False)
bars = axes[1].bar(zone_profit.index, zone_profit.values, color="#2980b9",
                   edgecolor="white")
axes[1].bar_label(bars, fmt="%.0f", fontsize=9, padding=2)
axes[1].set_title("The Middle zone produced the most profit in 2023")
axes[1].set_xlabel("Zone")
axes[1].set_ylabel("Profit (GHS)")
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].set_axisbelow(True)

fig.tight_layout()
plt.show()

---
## Key takeaways

- **Wrangling is shape work.** Cleaning fixes values; wrangling fixes the arrangement of tables.
- **State the grain of every table before you touch it.** One row is one *what*? Every later decision depends on the answer.
- `concat` **stacks** tables that are the same kind of thing. `merge` **matches** tables using a key.
- **A broken join usually does not raise an error.** It returns a plausible table full of plausible numbers.
- The three silent failures are **orphan keys** (rows vanish), **duplicate keys** (rows multiply), and **unclean keys** (nothing matches).
- `validate="m:1"` and `indicator=True` cost nothing and turn every one of those into a visible, checkable fact.
- **Compare the row count before and after every merge.** For an `m:1` left join it must not change.
- **You cannot join on a dirty key.** Clean first, then join.
- **Analyse long, present wide.** `melt()` to get usable; `pivot_table()` to get readable.
- `transform()` gives one value per original row; `agg()` gives one per group. Choose by the shape you need.
- **Sort before any window function.** `shift`, `cumsum`, and `rolling` all trust the row order.
- **Wrap the pipeline in a function** that validates its own output, so next month's data does not need this whole notebook re-run by hand.

## Further reading and the course path

### pandas documentation
- [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [`pandas.merge`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [`DataFrame.melt`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)
- [`DataFrame.pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html)
- [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Windowing operations](https://pandas.pydata.org/docs/user_guide/window.html)
- [`pandas.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) and [`pandas.qcut`](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html)

### Background
- [Tidy Data, by Hadley Wickham](https://vita.had.co.nz/papers/tidy-data.pdf) - the paper the three rules come from
- [Ghana 2021 Population and Housing Census](https://census2021.statsghana.gov.gh/) - source of the regional populations

### Practice notebooks for this topic
- [20a - Tidy data principles](topic20a_tidy_data_principles.ipynb)
- [20b - Stacking with concat](topic20b_stacking_with_concat.ipynb)
- [20c - Merging and joins](topic20c_merging_and_joins.ipynb)
- [20d - Merge hazards and validation](topic20d_merge_hazards_and_validation.ipynb)
- [20e - Reshaping with melt and pivot](topic20e_reshaping_melt_and_pivot.ipynb)
- [20f - Transforms and binning](topic20f_transforms_and_binning.ipynb)
- [20g - Wrangling practice](topic20g_wrangling_practice.ipynb)

### Assignment
- [Topic 20 assignment - retail wrangling](assignments/topic20_assignment_retail_wrangling.ipynb)

### Elsewhere in the course
- [Topic 17 - Data cleaning fundamentals](../data_cleaning_with_pandas/topic17_data_cleaning.ipynb)
- [Topic 18 - Basic data analysis and grouping](../basic_data_analysis/topic18_analysis_grouping.ipynb)
- [Topic 19 - Data gathering](../data_analysis_process/topic19_data_gathering.ipynb)
- [Submitting your work with GitHub](../data_analysis_process/topic19d_appendix_submitting_with_github.ipynb)